This notebook runs experiments on **Qwen/Qwen2.5-Math-1.5B**:

1. Zero-shot baseline on GSM8K  
2. SFT with reasoning traces  
3. GRPO with verified rewards


In [1]:
from pathlib import Path
import textwrap

REPO_URL = "https://github.com/lllcr6/cs336-a5.git"
REPO_DIR = Path("/content/assignment5-alignment")
BRANCH = "main"

WANDB_PROJECT = "cs336-assignment5-alignment"
WANDB_ENTITY = "chenrui6"

DRIVE_ROOT = Path("/content/drive/MyDrive/CS336-storage/cs336-assignment5")
RUN_ROOT = "qwen2p5-math-1p5b-gsm8k"

RESUME_FROM_CHECKPOINT = None

MODEL_ID = "Qwen/Qwen2.5-Math-1.5B"

GSM8K_TRAIN_PATH = REPO_DIR / "data" / "gsm8k" / "train.jsonl"
GSM8K_TEST_PATH = REPO_DIR / "data" / "gsm8k" / "test.jsonl"

In [2]:

from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:

import os
import subprocess

def run(cmd, cwd=None):
    print("+", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=cwd,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    return result

if REPO_DIR.exists():
    print(f"Refreshing existing repo at {REPO_DIR}")
    run(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    run(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH])
else:
    print(f"Cloning repo into {REPO_DIR}")
    run(["git", "clone", REPO_URL, str(REPO_DIR)])
    run(["git", "-C", str(REPO_DIR), "checkout", BRANCH])

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"])


Cloning repo into /content/assignment5-alignment
+ git clone https://github.com/lllcr6/cs336-a5.git /content/assignment5-alignment
Cloning into '/content/assignment5-alignment'...

+ git -C /content/assignment5-alignment checkout main
Already on 'main'
Your branch is up to date with 'origin/main'.

Working directory: /content/assignment5-alignment
+ git -C /content/assignment5-alignment log -1 --oneline
4d66c87 Clean up W&B metric names



CompletedProcess(args=['git', '-C', '/content/assignment5-alignment', 'log', '-1', '--oneline'], returncode=0, stdout='4d66c87 Clean up W&B metric names\n')

In [ ]:

import sys
import shlex
import subprocess
from pathlib import Path

PYTHON_VERSION = "3.12"
VENV_DIR = REPO_DIR / ".venv"

def bash(cmd, cwd=None, check=True):
    print("+", cmd)
    result = subprocess.run(
        ["bash", "-lc", cmd],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

bash(f"{sys.executable} -m pip install -U uv")
bash(f"cd {shlex.quote(str(REPO_DIR))} && uv python install {PYTHON_VERSION}")

if not VENV_DIR.exists():
    bash(f"cd {shlex.quote(str(REPO_DIR))} && uv venv --python {PYTHON_VERSION} .venv")
else:
    print(f"Skip: {VENV_DIR} already exists")

bash(f"cd {shlex.quote(str(REPO_DIR))} && uv sync --no-install-package flash-attn")
bash(f"cd {shlex.quote(str(REPO_DIR))} && uv sync", check=True)


+ /usr/bin/python3 -m pip install -U uv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 119.4 MB/s eta 0:00:00

+ cd /content/assignment5-alignment && uv python install 3.12
Installed Python 3.12.13 in 1.08s
 + cpython-3.12.13-linux-x86_64-gnu (python3.12)

+ cd /content/assignment5-alignment && uv venv --python 3.12 .venv
Using CPython 3.12.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate

+ cd /content/assignment5-alignment && uv sync --no-install-package flash-attn


In [6]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/assignment5-alignment")
TMP_RUN_DIR = REPO_DIR / "notebook_runs"
TMP_RUN_DIR.mkdir(parents=True, exist_ok=True)

def run_uv_python(script_text: str, script_name: str = "run_tmp.py"):
    script_path = TMP_RUN_DIR / script_name
    script_path.write_text(script_text, encoding="utf-8")

    env = os.environ.copy()

    # Remove inherited wandb service/socket state from the notebook process
    for key in [
        "WANDB_SERVICE",
        "WANDB_SERVICE_TRANSPORT",
        "WANDB_SERVICE_TOKEN",
        "WANDB_RUN_ID",
        "WANDB_RESUME",
        "WANDB_DIR",
    ]:
        env.pop(key, None)

    # Use a clean local wandb state for the subprocess
    env["WANDB_START_METHOD"] = "thread"

    cmd = f'cd "{REPO_DIR}" && uv run python "{script_path}"'
    print(f"$ {cmd}")
    result = subprocess.run(
        ["bash", "-lc", cmd],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout

In [7]:

import wandb

wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chenruiliu66 (chenrui6) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
COMMON_SCRIPT = rf'''
from pathlib import Path
import json
import wandb

from cs336_alignment.drgrpo_grader import extract_answer, grade
import re

REPO_DIR = Path("{REPO_DIR}")
MODEL_ID = "{MODEL_ID}"
WANDB_PROJECT = "{WANDB_PROJECT}"
WANDB_ENTITY = "{WANDB_ENTITY}"
GSM8K_TRAIN_PATH = Path("{GSM8K_TRAIN_PATH}")
GSM8K_TEST_PATH = Path("{GSM8K_TEST_PATH}")
DRIVE_ROOT = Path("{DRIVE_ROOT}")
RUN_ROOT = "{RUN_ROOT}"
RESUME_FROM_CHECKPOINT = {repr(RESUME_FROM_CHECKPOINT)}

def slugify(text):
    return str(text).replace("/", "-").replace(" ", "-").replace(".", "-")

def dataset_slug(path):
    p = Path(path)
    return f"{{p.parent.name}}-{{p.stem}}"

def build_run_name(method, model_id, dataset_path, extra=""):
    model_tag = slugify(model_id)
    data_tag = dataset_slug(dataset_path)
    parts = [method, data_tag, model_tag]
    if extra:
        parts.append(slugify(extra))
    return "__".join(parts)

def build_run_dirs(run_name, need_checkpoint=True):
    local_output_dir = REPO_DIR / "outputs" / run_name
    local_eval_dir = local_output_dir / "eval"
    local_wandb_dir = local_output_dir / "wandb"

    local_output_dir.mkdir(parents=True, exist_ok=True)
    local_eval_dir.mkdir(parents=True, exist_ok=True)
    local_wandb_dir.mkdir(parents=True, exist_ok=True)

    dirs = {{
        "local_output_dir": local_output_dir,
        "local_eval_dir": local_eval_dir,
        "local_wandb_dir": local_wandb_dir,
    }}

    if need_checkpoint:
        local_checkpoint_dir = local_output_dir / "checkpoints"
        local_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        dirs["local_checkpoint_dir"] = local_checkpoint_dir

    return dirs

def extract_final_answer(text: str) -> str | None:
    if text is None:
        return None
    text = str(text).strip()

    boxed = extract_answer(text)
    if boxed is not None:
        return boxed.strip()

    if "####" in text:
        candidate = text.split("####")[-1].strip()
        if candidate:
            return candidate

    m = re.search(r"(?i)(?:final answer|answer)\s*[:\-]\s*([^\n]+)", text)
    if m:
        return m.group(1).strip()

    number_matches = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if number_matches:
        return number_matches[-1].replace(",", "")

    return None

def verified_gsm8k_reward(response, ground_truth):
    pred = extract_final_answer(response)
    gt = extract_final_answer(ground_truth) or str(ground_truth)

    if pred is None:
        reward = 0.0
    else:
        reward = 1.0 if grade(pred, gt) else 0.0

    return {{
        "reward": reward,
        "format_reward": 1.0 if pred is not None else 0.0,
        "answer_reward": reward,
    }}
'''

## Experiment 1: Zero-shot baseline

In [13]:
def make_baseline_script():
    return COMMON_SCRIPT + rf'''
from cs336_alignment import run_zero_shot_baseline
from cs336_alignment.config import EvalConfig

BASELINE_NUM_EXAMPLES = 512
BASELINE_MAX_TOKENS = 512
BASELINE_BATCH_SIZE = 8
BASELINE_TEMPERATURE = 1.0
BASELINE_TOP_P = 1.0

baseline_run_name = build_run_name("baseline", MODEL_ID, GSM8K_TEST_PATH, "zero-shot")
baseline_dirs = build_run_dirs(baseline_run_name, need_checkpoint=False)

baseline_eval_config = EvalConfig(
    output_dir=baseline_dirs["local_eval_dir"],
    batch_size=BASELINE_BATCH_SIZE,
    temperature=BASELINE_TEMPERATURE,
    top_p=BASELINE_TOP_P,
    max_tokens=BASELINE_MAX_TOKENS,
    num_examples=BASELINE_NUM_EXAMPLES,
)

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=baseline_run_name,
    dir=str(baseline_dirs["local_wandb_dir"]),
    tags=["assignment5", "baseline", "zero-shot", dataset_slug(GSM8K_TEST_PATH), slugify(MODEL_ID)],
    config={{
        "method": "baseline",
        "model_id": MODEL_ID,
        "dataset_path": str(GSM8K_TEST_PATH),
        "batch_size": BASELINE_BATCH_SIZE,
        "temperature": BASELINE_TEMPERATURE,
        "top_p": BASELINE_TOP_P,
        "max_tokens": BASELINE_MAX_TOKENS,
        "num_examples": BASELINE_NUM_EXAMPLES,
    }},
    resume="allow",
    settings=wandb.Settings(start_method="thread"),
)

baseline_result = run_zero_shot_baseline(
    model_id=MODEL_ID,
    dataset_path=GSM8K_TEST_PATH,
    reward_fn=verified_gsm8k_reward,
    eval_config=baseline_eval_config,
)

summary = baseline_result["results"]["summary"]
payload = {{
    "results/summary/num_examples": summary.get("num_examples"),
    "results/summary/reward_mean": summary.get("reward_mean"),
    "results/summary/answer_reward_mean": summary.get("answer_reward_mean"),
    "results/summary/format_reward_mean": summary.get("format_reward_mean"),
}}
wandb.log({{k: v for k, v in payload.items() if v is not None}})


wandb.finish()
print(baseline_result)
'''

baseline_script = make_baseline_script()
run_uv_python(baseline_script, "run_baseline.py")

$ cd "/content/assignment5-alignment" && uv run python "/content/assignment5-alignment/notebook_runs/run_baseline.py"
wandb: Currently logged in as: chenruiliu66 (chenrui6) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: creating run
wandb: Tracking run with wandb version 0.19.11
wandb: Run data is saved locally in /content/assignment5-alignment/outputs/baseline__gsm8k-test__Qwen-Qwen2-5-Math-1-5B__zero-shot/wandb/wandb/run-20260326_180053-yptqjs5k
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run baseline__gsm8k-test__Qwen-Qwen2-5-Math-1-5B__zero-shot
wandb: ⭐️ View project at https://wandb.ai/chenrui6/cs336-assignment5-alignment
wandb: 🚀 View run at https://wandb.ai/chenrui6/cs336-assignment5-alignment/runs/yptqjs5k
INFO 03-26 18:00:58 __init__.py:190] Automatically detected platform cuda.
INFO 03-26 18:01:09 config.py:542] This model supports multiple tasks: {'score', 'generate', 'embed', 'reward', 'classify'}. Defaulting to 'generate'.


'wandb: Currently logged in as: chenruiliu66 (chenrui6) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin\nwandb: creating run\nwandb: Tracking run with wandb version 0.19.11\nwandb: Run data is saved locally in /content/assignment5-alignment/outputs/baseline__gsm8k-test__Qwen-Qwen2-5-Math-1-5B__zero-shot/wandb/wandb/run-20260326_180053-yptqjs5k\nwandb: Run `wandb offline` to turn off syncing.\nwandb: Syncing run baseline__gsm8k-test__Qwen-Qwen2-5-Math-1-5B__zero-shot\nwandb: ⭐️ View project at https://wandb.ai/chenrui6/cs336-assignment5-alignment\nwandb: 🚀 View run at https://wandb.ai/chenrui6/cs336-assignment5-alignment/runs/yptqjs5k\nINFO 03-26 18:00:58 __init__.py:190] Automatically detected platform cuda.\nINFO 03-26 18:01:09 config.py:542] This model supports multiple tasks: {\'score\', \'generate\', \'embed\', \'reward\', \'classify\'}. Defaulting to \'generate\'.\nINFO 03-26 18:01:09 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model

## Experiment 2: SFT with reasoning traces

In [ ]:
def make_sft_script():
    return COMMON_SCRIPT + rf'''
from cs336_alignment import run_sft_training
from cs336_alignment.config import CheckpointConfig, DriveSyncConfig, EvalConfig, SFTConfig, WandbConfig

SFT_TRAIN_STEPS = 400
SFT_LR = 1e-5
SFT_BATCH_SIZE = 16
SFT_GRAD_ACCUM = 4

SFT_EVAL_EVERY_STEPS = 50
SFT_SAVE_EVERY_STEPS = 150

SFT_EVAL_BATCH_SIZE = 8
SFT_EVAL_MAX_TOKENS = 512
SFT_EVAL_NUM_EXAMPLES = 512
SFT_EVAL_TEMPERATURE = 1.0
SFT_EVAL_TOP_P = 1.0

sft_run_name = build_run_name("sft", MODEL_ID, GSM8K_TRAIN_PATH, "reasoning-traces")
sft_dirs = build_run_dirs(sft_run_name)

sft_config = SFTConfig(
    train_steps=SFT_TRAIN_STEPS,
    learning_rate=SFT_LR,
    train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    normalize_constant=1.0,
    eval_every_steps=SFT_EVAL_EVERY_STEPS,
    save_every_steps=SFT_SAVE_EVERY_STEPS,
    wandb=WandbConfig(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        run_name=sft_run_name,
        tags=["assignment5", "sft", dataset_slug(GSM8K_TRAIN_PATH), slugify(MODEL_ID), "reasoning-traces"],
        log_dir=sft_dirs["local_wandb_dir"],
    ),
    checkpoint=CheckpointConfig(
        output_dir=sft_dirs["local_checkpoint_dir"],
        save_every_steps=SFT_SAVE_EVERY_STEPS,
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    ),
    drive_sync=DriveSyncConfig(
        enabled=True,
        drive_root=DRIVE_ROOT / RUN_ROOT,
        per_run_subdir=sft_run_name,
    ),
)

sft_eval_config = EvalConfig(
    output_dir=sft_dirs["local_eval_dir"],
    batch_size=SFT_EVAL_BATCH_SIZE,
    temperature=SFT_EVAL_TEMPERATURE,
    top_p=SFT_EVAL_TOP_P,
    max_tokens=SFT_EVAL_MAX_TOKENS,
    num_examples=SFT_EVAL_NUM_EXAMPLES,
)

wandb_extra = {{
    "train_steps": SFT_TRAIN_STEPS,
    "learning_rate": SFT_LR,
    "train_batch_size": SFT_BATCH_SIZE,
    "gradient_accumulation_steps": SFT_GRAD_ACCUM,
    "eval_every_steps": SFT_EVAL_EVERY_STEPS,
    "save_every_steps": SFT_SAVE_EVERY_STEPS,
    "eval_batch_size": SFT_EVAL_BATCH_SIZE,
    "eval_temperature": SFT_EVAL_TEMPERATURE,
    "eval_top_p": SFT_EVAL_TOP_P,
    "eval_max_tokens": SFT_EVAL_MAX_TOKENS,
    "eval_num_examples": SFT_EVAL_NUM_EXAMPLES,
}}

sft_result = run_sft_training(
    model_id=MODEL_ID,
    dataset_path=GSM8K_TRAIN_PATH,
    validation_dataset_path=GSM8K_TEST_PATH,
    config=sft_config,
    eval_config=sft_eval_config,
    reward_fn=verified_gsm8k_reward,
)

print(json.dumps(wandb_extra, indent=2, ensure_ascii=False))
print(json.dumps({{k: v for k, v in sft_result.items() if k != "loss_history"}}, indent=2, ensure_ascii=False))
'''

sft_script = make_sft_script()
run_uv_python(sft_script, "run_sft.py")

$ cd "/content/assignment5-alignment" && uv run python "/content/assignment5-alignment/notebook_runs/run_sft.py"


## Experiment 3: GRPO with verified rewards

In [ ]:
def make_grpo_script():
    return COMMON_SCRIPT + rf'''
from cs336_alignment import run_grpo_training
from cs336_alignment.config import CheckpointConfig, DriveSyncConfig, EvalConfig, GRPOConfig, WandbConfig

GRPO_TRAIN_STEPS = 50
GRPO_LR = 5e-6
GRPO_ROLLOUT_BATCH_SIZE = 16
GRPO_TRAIN_BATCH_SIZE = 16
GRPO_GROUP_SIZE = 4
GRPO_GRAD_ACCUM = 8
GRPO_EPOCHS_PER_ROLLOUT = 1

GRPO_EVAL_EVERY_STEPS = 10
GRPO_SAVE_EVERY_STEPS = 100

GRPO_EVAL_BATCH_SIZE = 8
GRPO_EVAL_MAX_TOKENS = 512
GRPO_EVAL_TEMPERATURE = 1.0
GRPO_EVAL_TOP_P = 1.0
GRPO_EVAL_NUM_EXAMPLES = 512

grpo_run_name = build_run_name("grpo", MODEL_ID, GSM8K_TRAIN_PATH, "verified-reward")
grpo_dirs = build_run_dirs(grpo_run_name)

grpo_config = GRPOConfig(
    train_steps=GRPO_TRAIN_STEPS,
    learning_rate=GRPO_LR,
    rollout_batch_size=GRPO_ROLLOUT_BATCH_SIZE,
    group_size=GRPO_GROUP_SIZE,
    train_batch_size=GRPO_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRPO_GRAD_ACCUM,
    epochs_per_rollout_batch=GRPO_EPOCHS_PER_ROLLOUT,
    eval_every_steps=GRPO_EVAL_EVERY_STEPS,
    save_every_steps=GRPO_SAVE_EVERY_STEPS,
    wandb=WandbConfig(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        run_name=grpo_run_name,
        tags=["assignment5", "grpo", dataset_slug(GSM8K_TRAIN_PATH), slugify(MODEL_ID), "verified-reward"],
        log_dir=grpo_dirs["local_wandb_dir"],
    ),
    checkpoint=CheckpointConfig(
        output_dir=grpo_dirs["local_checkpoint_dir"],
        save_every_steps=GRPO_SAVE_EVERY_STEPS,
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    ),
    drive_sync=DriveSyncConfig(
        enabled=True,
        drive_root=DRIVE_ROOT / RUN_ROOT,
        per_run_subdir=grpo_run_name,
    ),
)

grpo_eval_config = EvalConfig(
    output_dir=grpo_dirs["local_eval_dir"],
    batch_size=GRPO_EVAL_BATCH_SIZE,
    temperature=GRPO_EVAL_TEMPERATURE,
    top_p=GRPO_EVAL_TOP_P,
    max_tokens=GRPO_EVAL_MAX_TOKENS,
    num_examples=GRPO_EVAL_NUM_EXAMPLES,
)

wandb_extra = {{
    "train_steps": GRPO_TRAIN_STEPS,
    "learning_rate": GRPO_LR,
    "rollout_batch_size": GRPO_ROLLOUT_BATCH_SIZE,
    "group_size": GRPO_GROUP_SIZE,
    "train_batch_size": GRPO_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRPO_GRAD_ACCUM,
    "epochs_per_rollout_batch": GRPO_EPOCHS_PER_ROLLOUT,
    "eval_every_steps": GRPO_EVAL_EVERY_STEPS,
    "save_every_steps": GRPO_SAVE_EVERY_STEPS,
    "eval_batch_size": GRPO_EVAL_BATCH_SIZE,
    "eval_temperature": GRPO_EVAL_TEMPERATURE,
    "eval_top_p": GRPO_EVAL_TOP_P,
    "eval_max_tokens": GRPO_EVAL_MAX_TOKENS,
    "eval_num_examples": GRPO_EVAL_NUM_EXAMPLES,
}}

grpo_result = run_grpo_training(
    model_id=MODEL_ID,
    train_dataset_path=GSM8K_TRAIN_PATH,
    validation_dataset_path=GSM8K_TEST_PATH,
    reward_fn=verified_gsm8k_reward,
    config=grpo_config,
    eval_config=grpo_eval_config,
)

print(json.dumps(wandb_extra, indent=2, ensure_ascii=False))
print(json.dumps({{k: v for k, v in grpo_result.items() if k != "loss_history"}}, indent=2, ensure_ascii=False))
'''

grpo_script = make_grpo_script()
run_uv_python(grpo_script, "run_grpo.py")

$ cd "/content/assignment5-alignment" && uv run python "/content/assignment5-alignment/notebook_runs/run_grpo.py"


## Run order

Run the setup cells first, then run the four experiment cells one by one.
